In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import json
import math
import multiprocessing
from collections import defaultdict

def worker_validate_chunk(args):
    """
    각 프로세스가 파일의 [start_byte, end_byte] 구간만 읽으면서
    1) JSON 파싱 오류(예: 구문 에러, dict가 아닌 객체) 체크
    2) key별 타입(type) 조사

    반환값: {
        "type_map": {key: [타입이름, ...], ...},
        "parse_errors": [오류 라인 원문 예시 (최대 max_errors_per_worker개)], 
        "lines_processed": 처리한 라인 수,
        "errors_count": 파싱 오류가 난 라인 수
    }
    """
    file_path, start_byte, end_byte, max_errors_per_worker = args

    local_type_map = defaultdict(set)
    parse_errors = []
    lines_processed = 0
    errors_count = 0

    with open(file_path, "rb") as f:
        # 1) 시작 바이트로 이동
        f.seek(start_byte)

        # 2) start_byte가 0이 아니면, 해당 바이트가 중간 라인일 수 있으므로
        #    그 라인을 통으로 건너뛰기 위해 readline() 한 번 호출
        if start_byte != 0:
            f.readline()

        # 3) 이제부터 end_byte를 넘지 않는 한 읽어들이며 처리
        while True:
            pos = f.tell()
            if pos > end_byte:
                break

            raw = f.readline()
            if not raw:
                # 파일 끝에 도달
                break

            try:
                line_str = raw.decode("utf-8").strip()
            except:
                # 디코딩 자체가 안 되는 바이너리 조각일 경우, 건너뛰기
                continue

            if line_str == "":
                # 빈 줄도 오류로 간주
                errors_count += 1
                if len(parse_errors) < max_errors_per_worker:
                    parse_errors.append("[빈 줄]")
                continue

            lines_processed += 1

            try:
                obj = json.loads(line_str)
            except json.JSONDecodeError:
                # JSON 문법 오류
                errors_count += 1
                if len(parse_errors) < max_errors_per_worker:
                    parse_errors.append(line_str)
                continue

            if not isinstance(obj, dict):
                # 최상위 객체가 dict가 아닌 경우
                errors_count += 1
                if len(parse_errors) < max_errors_per_worker:
                    parse_errors.append(line_str)
                continue

            # key별 타입(type) 수집
            for k, v in obj.items():
                local_type_map[k].add(type(v).__name__)

    # Pickle 가능한 형식으로 변환 (set → list)
    serializable_map = {k: list(v) for k, v in local_type_map.items()}

    return {
        "type_map": serializable_map,
        "parse_errors": parse_errors,
        "lines_processed": lines_processed,
        "errors_count": errors_count,
    }


def parallel_validate_jsonl(file_path: str, num_workers: int = None, max_errors_per_worker: int = 5):
    """
    1) file_path 파일 크기를 구해서, 'num_workers'개로 바이트 구간을 나눔  
    2) multiprocessing.Pool을 이용해 worker_validate_chunk를 병렬 호출  
    3) 각각의 결과(type_map, parse_errors 등)를 모아서 최종 요약 출력

    Args:
        file_path (str): 검사할 JSONL 파일 전체 경로
        num_workers (int or None): 사용할 프로세스 수. None이면 cpu_count() 사용
        max_errors_per_worker (int): 각 프로세스가 최대 몇 개의 오류 라인 원문을 모아올지
    """
    if num_workers is None:
        num_workers = multiprocessing.cpu_count()

    # 1) 파일 사이즈(바이트) 구하기
    try:
        file_size = os.path.getsize(file_path)
    except OSError as e:
        print(f"❌ 파일을 찾을 수 없습니다: {file_path}")
        return

    # 2) 각 워커가 담당할 바이트 구간 크기 (정수 나눗셈, 맨 마지막 워커는 남은 전부)
    base_chunk = file_size // num_workers
    byte_ranges = []

    for i in range(num_workers):
        start_byte = i * base_chunk
        if i < num_workers - 1:
            end_byte = (i + 1) * base_chunk - 1
        else:
            end_byte = file_size - 1
        byte_ranges.append((start_byte, end_byte))

    # 3) Pool에 넘길 인자 리스트 구성
    task_args = []
    for (start, end) in byte_ranges:
        task_args.append((file_path, start, end, max_errors_per_worker))

    print(f"▶️  총 파일 크기: {file_size / (1024**3):.2f} GB")
    print(f"▶️  CPU 코어: {multiprocessing.cpu_count()}개, 사용할 워커: {num_workers}개")
    print(f"▶️  각 워커당 대략 {base_chunk / (1024**3):.2f} GB 바이트 범위를 처리합니다.\n")

    # 4) 병렬 처리
    with multiprocessing.Pool(num_workers) as pool:
        results = pool.map(worker_validate_chunk, task_args)

    # 5) 결과 종합
    final_type_map = defaultdict(set)
    all_parse_errors = []
    total_lines = 0
    total_error_count = 0

    for res in results:
        # 5-1) key별 타입을 집합 형태로 합치기
        for k, type_list in res["type_map"].items():
            final_type_map[k].update(type_list)

        # 5-2) parse_errors 모아두기 (최대 num_workers * max_errors_per_worker개)
        all_parse_errors.extend(res["parse_errors"])

        # 5-3) 총 라인 수와 총 오류 수 집계
        total_lines += res["lines_processed"]
        total_error_count += res["errors_count"]

    # 6) 최종 요약 정보 출력
    print("=" * 80)
    print(f"✔️  병렬 검사 완료: {file_path}")
    print(f"  • 전체(처리된) 라인 수        : {total_lines}")
    print(f"  • 총 파싱 오류(문법 / dict 아님) 건수 : {total_error_count}")
    if all_parse_errors:
        print(f"  • 오류 라인 원문 예시 (최대 {len(all_parse_errors)}개):")
        for i, err in enumerate(all_parse_errors[: min(len(all_parse_errors), num_workers * max_errors_per_worker)]):
            print(f"    {i+1}. {err[:200]}{'…' if len(err) > 200 else ''}")
    else:
        print("  • 파싱 오류가 전혀 없습니다.")

    print("\n📊 키(key)별 관측된 데이터 타입(최대 3개만 표시)")
    for k, types in sorted(final_type_map.items()):
        types_list = list(types)
        preview = types_list[:3]
        more = f", …(+{len(types_list)-3})" if len(types_list) > 3 else ""
        print(f"  • {k}: {preview}{more}")

    print("\n⚠️ 스키마 불일치 가능성이 있는 키(타입이 2개 이상 관측된 경우):")
    mixed_keys = [k for k, types in final_type_map.items() if len(types) > 1]
    if mixed_keys:
        for k in mixed_keys:
            print(f"    – {k}: {final_type_map[k]}")
    else:
        print("    – 모든 키가 일관된 타입을 가집니다.")
    print("=" * 80 + "\n")


if __name__ == "__main__":
    # ────────────────────────────────────────────────────────────
    # 1) 검사할 JSONL 파일 경로와 멀티프로세싱 옵션을 직접 이곳에서 설정
    # ────────────────────────────────────────────────────────────
    jsonl_path = "/home/remote/Ai_Capstone_Project/conversation_1.7G.jsonl"
    # 원본 데이터가 너무 클 경우(32 GB), 미리 일부 라인만 테스트해 보고 싶다면
    # → sample_limit 기능을 사용하거나, 아래처럼 파일을 분할해서 검사하세요.
    #   (아래 스크립트는 파일 통째로 검사합니다.)

    # 병렬 워커 개수 (None으로 두면 CPU 코어 수를 자동으로 사용)
    num_workers = None

    # 워커 당 최대 수집할 “파싱 오류 라인 원문” 개수
    max_errors_per_worker = 5

    # ────────────────────────────────────────────────────────────
    # 2) 병렬 검사 함수 호출
    # ────────────────────────────────────────────────────────────
    parallel_validate_jsonl(
        file_path=jsonl_path,
        num_workers=num_workers,
        max_errors_per_worker=max_errors_per_worker
    )


▶️  총 파일 크기: 1.65 GB
▶️  CPU 코어: 32개, 사용할 워커: 32개
▶️  각 워커당 대략 0.05 GB 바이트 범위를 처리합니다.

✔️  병렬 검사 완료: /home/remote/Ai_Capstone_Project/conversation_1.7G.jsonl
  • 전체(처리된) 라인 수        : 24998704
  • 총 파싱 오류(문법 / dict 아님) 건수 : 24998704
  • 오류 라인 원문 예시 (최대 160개):
    1. {
    2. "instruction": "배경: 고대  도시: 모헨조다로    플레이어: 이지수(사이버 해커, 평민)  NPC: 시봉우리(섭정, 정령, 적대)) 상황: 이지수(플레이어)이(가) 시봉우리(NPC)에게 인사를 청하는 상황",
    3. "input": "안녕하세요",
    4. "output": "안녕하세요."
    5. }
    6. "input": "당신은 어떤 성별로 살아가고 있나요?",
    7. "output": "나는 자랑스러운 여자야."
    8. }
    9. {
    10. "instruction": "배경: 현대  도시: 스카이라인    플레이어: 이지수(사이버 해커, 평민)  NPC: 양별(후작, 귀족, 중립)) 상황: 이지수(플레이어)이(가) 양별(NPC)에게 성별을 묻는 상황",
    11. "input": "출신지는 어디야?",
    12. "output": "언더크립트는 나의 뿌리와도 같은 곳이야."
    13. }
    14. {
    15. "instruction": "배경: 판타지  도시: 언더크립트    플레이어: 이지수(사이버 해커, 평민)  NPC: 범솔돌(사제, 신격, 중립)) 상황: 이지수(플레이어)이(가) 범솔돌(NPC)에게 출신 도시를 묻는 상황",
    16. "input": "도시는 어디에 위치해?",
    17. "output": "고향은 브라이트브룩야."
    18. }
    19. {
    

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import json
import os
from multiprocessing import Pool, cpu_count

def try_fix_braces(obj_text: str) -> str:
    """
    obj_text에 '{'로 시작하거나 '}'로 끝나지 않는 부분이 있으면
    각각 앞/뒤에 중괄호를 붙여서 간단히 보강해 줍니다.
    - 예: '"input": "..." , "output": "..." }'  →  '{ "input": "...", "output": "..." }'
    - 예: '{ "a": 1, "b": 2'                 →  '{ "a": 1, "b": 2 }'
    """
    txt = obj_text.strip()

    # 1) 맨 앞이 '{'가 아니라면 앞에 '{' 추가
    if not txt.startswith("{"):
        txt = "{" + txt

    # 2) 맨 끝이 '}'가 아니라면 뒤에 '}' 추가
    if not txt.endswith("}"):
        txt = txt + "}"

    return txt


def parse_to_single_line(obj_text: str) -> str or None:
    """
    Worker 함수: 멀티라인 JSON(obj_text)을 받아서
    1) json.loads(obj_text)로 파싱을 시도
    2) 실패 시, try_fix_braces(obj_text)로 중괄호 보강 후 다시 파싱 시도
    3) 두 번째도 실패하면 None 반환 → 해당 블록은 건너뜁니다.

    반환: 정상 변환된 한 줄(single-line) JSON 문자열 or None
    """
    try:
        obj = json.loads(obj_text)
        return json.dumps(obj, ensure_ascii=False)
    except json.JSONDecodeError:
        # 중괄호가 빠진 경우를 간단히 보강해 본다
        fixed_text = try_fix_braces(obj_text)
        try:
            obj = json.loads(fixed_text)
            return json.dumps(obj, ensure_ascii=False)
        except json.JSONDecodeError:
            # 여전히 파싱 불가 → None 리턴
            return None


def iter_multiline_objects(file_path: str):
    """
    Generator: 파일을 한 줄씩 읽으며
    '{' / '}' 개수를 추적하여 “한 JSON 오브젝트가 완성된 시점”마다
    그동안 모인 여러 줄(buffer_lines)을 하나의 문자열(obj_text)로 합쳐 yield 합니다.

    예시 yield:
       '{\n"instruction": "...",\n"input": "...",\n"output": "..." \n}'
    """
    buffer_lines = []
    brace_balance = 0  # '{' 개수 - '}' 개수

    with open(file_path, "r", encoding="utf-8") as fin:
        for raw_line in fin:
            line = raw_line.rstrip("\n")

            # 1) 버퍼가 비어 있고, 이 줄이 '{'로 시작하지 않으면 → 오브젝트 시작 아님, 스킵
            if brace_balance == 0 and not line.strip().startswith("{"):
                continue

            buffer_lines.append(line)
            brace_balance += line.count("{")
            brace_balance -= line.count("}")

            # 2) 중괄호 균형이 0이 되면 “하나의 오브젝트 블록” 완성
            if brace_balance == 0 and buffer_lines:
                obj_text = "\n".join(buffer_lines)
                yield obj_text
                buffer_lines = []
                brace_balance = 0


def fix_multiline_jsonl_parallel(orig_path: str, fixed_path: str, num_workers: int = None):
    """
    멀티라인 JSONL → 한 줄 JSONL 병렬 변환 함수

    Args:
        orig_path   (str): 원본 멀티라인 JSONL 파일 경로
        fixed_path  (str): 한 줄(JSON)으로 변환된 결과 파일 경로
        num_workers (int): 병렬 워커 수. None이면 CPU 코어 수를 자동으로 사용
    """
    # 0) 파일 존재 여부 확인
    if not os.path.isfile(orig_path):
        print(f"❌ 파일을 찾을 수 없습니다: {orig_path}")
        return

    # 1) CPU 코어 수만큼 워커 수 자동 지정
    if num_workers is None:
        num_workers = cpu_count()

    os.makedirs(os.path.dirname(fixed_path), exist_ok=True)

    obj_count = 0  # 정상 변환된 오브젝트 수
    bad_count = 0  # 파싱 실패로 건너뛴 오브젝트 수

    # 2) 멀티프로세싱 풀 생성 (워커 수 = num_workers)
    pool = Pool(processes=num_workers)

    print(f"▶️  병렬 변환 시작: {orig_path}")
    print(f"   • 사용할 워커 개수: {num_workers}\n")

    # 3) Reader + Worker + Writer 연결
    #    - Reader: iter_multiline_objects(orig_path) → obj_text 여러 줄 문자열을 순차 생성
    #    - Worker: parse_to_single_line(obj_text) → 한 줄(JSON) 문자열 or None 반환
    #    - Writer: 반환된 결과를 순서대로 파일에 기록
    with open(fixed_path, "w", encoding="utf-8") as fout:
        for single_line in pool.imap(parse_to_single_line,
                                     iter_multiline_objects(orig_path),
                                     chunksize=100):
            if single_line is None:
                bad_count += 1
            else:
                fout.write(single_line + "\n")
                obj_count += 1

    pool.close()
    pool.join()

    total_objs = obj_count + bad_count

    print("\n✅ 병렬 변환 완료")
    print(f"  • 생성된 JSON 오브젝트 수          : {total_objs}")
    print(f"     – 정상 변환된 오브젝트(한 줄 JSON): {obj_count}")
    print(f"     – 파싱 실패로 건너뛴 오브젝트    : {bad_count}")
    print(f"  • 결과 파일 경로                    : {fixed_path}\n")


if __name__ == "__main__":
    # ────────────────────────────────────────────────────────────
    # 1) 원본 멀티라인 JSONL 파일 경로 지정
    # ────────────────────────────────────────────────────────────
    orig_jsonl = "/home/remote/Ai_Capstone_Project/conversation_1.7G.jsonl"
    # (예: "/home/remote/Ai_Capstone_Project/data_100M.jsonl")

    # ────────────────────────────────────────────────────────────
    # 2) 변환 결과(한 줄 JSONL) 파일 경로 지정
    # ────────────────────────────────────────────────────────────
    fixed_jsonl = "/home/remote/Ai_Capstone_Project/conversation_1.7G_singleline.jsonl"

    # ────────────────────────────────────────────────────────────
    # 3) 워커 프로세스 개수 (None이면 CPU 코어 수 자동 사용)
    # ────────────────────────────────────────────────────────────
    num_workers = None

    fix_multiline_jsonl_parallel(orig_jsonl, fixed_jsonl, num_workers)


▶️  병렬 변환 시작: /home/remote/Ai_Capstone_Project/conversation_1.7G.jsonl
   • 사용할 워커 개수: 32


✅ 병렬 변환 완료
  • 생성된 JSON 오브젝트 수          : 4999741
     – 정상 변환된 오브젝트(한 줄 JSON): 4999741
     – 파싱 실패로 건너뛴 오브젝트    : 0
  • 결과 파일 경로                    : /home/remote/Ai_Capstone_Project/conversation_1.7G_singleline.jsonl

